# Enriching the apartment data with open data (municipality data)

**This Jupyter notebook does the following:**

- Imports the cleaned apartment data with the renovation details (`Data/apartments_pre1990_renovation_details.csv`, from `05_renovation_details.ipynb`)
- Assigns each apartment to its **municipality** (BFS number) with the official postcode directory of swisstopo
- Adds the coordinates (latitude/longitude of the postcode area), the canton and the language region
- Joins **municipality-level open data** (SFSO Regionalporträts 2021: population, population density, share of foreigners, employment, mean taxable income) using `.merge()`
- Removes listings that are probably new buildings
- Saves the enriched data to `Data/apartments_pre1990_enriched.csv`

The structure follows `Week_09/combining_and_organizing_data.ipynb`.

**Open data sources:**

| File | Source | Content |
|---|---|---|
| `AMTOVZ_CSV_WGS84.csv` | swisstopo, *Amtliches Ortschaftenverzeichnis mit Postleitzahl und Perimeter* (data.geo.admin.ch, download 26.11.2026) | postcode, locality, municipality name, BFS number, canton, address share, coordinates (WGS84), language |
| `municipality_data_with_taxable_income.xlsx` | SFSO, *Regionalporträts 2021: Kennzahlen aller Gemeinden* (from `Week_09/Data`) | population, population density, share of foreigners, employment, mean taxable income |

## Libraries and settings

In [1]:
# Libraries
import os
import re
import unicodedata
import numpy as np
import pandas as pd

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Show current working directory
print(os.getcwd())

c:\Users\DELL\Documents\applied_research_methods\Project_Topic1


## Importing the apartment data

In [2]:
df = pd.read_csv('./Data/apartments_pre1990_renovation_details.csv', sep=',', encoding='utf-8')

print('Dimension:', df.shape)
df[['source', 'address', 'zip', 'locality', 'rooms', 'area_clean', 'price', 'renovation_group']].head()

Dimension: (733, 35)


,source,address,zip,locality,rooms,area_clean,price,renovation_group
0,immoscout24,"Croix Fédérale 44, 2300 La Chaux-de-Fonds",2300,La Chaux-de-Fonds,4.5,83.0,1485.0,general renovation
1,immoscout24,"Grüttbachstr. 6, 4542 Luterbach",4542,Luterbach,4.5,108.0,1990.0,general renovation
2,immoscout24,"Neuhofstr. 20, 5600 Lenzburg",5600,Lenzburg,4.5,80.0,1760.0,energy renovation
3,immoscout24,"Aathalstr. 19, 8610 Uster",8610,Uster,5.5,116.0,2220.0,not renovated
4,immoscout24,"Rte de Bertigny 2/4, 1700 Fribourg",1700,Fribourg,2.0,52.0,1275.0,general renovation


## Assign each apartment to its municipality

### Reading the postcode directory of swisstopo

One postcode can belong to **several** municipalities (e.g. postcode 8610 Uster). The column `Adressenanteil` shows which share of the addresses of the postcode lies in each municipality.

In [3]:
df_plz = pd.read_csv('./Data/AMTOVZ_CSV_WGS84.csv', sep=';', encoding='utf-8-sig')

# Address share as a number, e.g. '100 %' -> 100.0
df_plz['address_share'] = df_plz['Adressenanteil'].str.replace('%', '').str.strip().astype(float)

# Rename columns
df_plz = df_plz.rename(columns={'PLZ4': 'zip',
                                'Ortschaftsname': 'plz_locality',
                                'Gemeindename': 'municipality',
                                'BFS-Nr': 'bfs_number',
                                'Kantonskürzel': 'canton',
                                'E': 'lon',
                                'N': 'lat',
                                'Sprache': 'language'})

print('Rows:', df_plz.shape[0], '| different postcodes:', df_plz['zip'].nunique())
df_plz[['zip', 'plz_locality', 'municipality', 'bfs_number', 'canton', 'address_share', 'lat', 'lon', 'language']].head()

Rows: 5717 | different postcodes: 3190


,zip,plz_locality,municipality,bfs_number,canton,address_share,lat,lon,language
0,1000,Lausanne 25,Lausanne,5586,VD,100.0,46.559589,6.682261,fr
1,1000,Lausanne 26,Lausanne,5586,VD,100.0,46.554570,6.695932,fr
2,1000,Lausanne 27,Lausanne,5586,VD,100.0,46.542070,6.681672,fr
3,1003,Lausanne,Lausanne,5586,VD,100.0,46.520568,6.631864,fr
4,1004,Lausanne,Lausanne,5586,VD,100.0,46.528291,6.618649,fr


### Matching rule

1. **Postcode and locality** are identical (the locality is compared in lower case without accents, spaces and punctuation, e.g. *"Biel/Bienne"* = *"bielbienne"*). If several municipalities fit, the one with the largest address share is used.
2. If the locality is not found (e.g. *"Biel-Vingelz"*), the municipality with the **largest address share of the postcode** is used.

In [4]:
def norm_name(s):
    s = unicodedata.normalize('NFKD', str(s).lower())
    s = ''.join(c for c in s if not unicodedata.combining(c))
    return re.sub(r'[^a-z]', '', s)

df['loc_key'] = df['locality'].map(norm_name)
df_plz['loc_key'] = df_plz['plz_locality'].map(norm_name)

plz_cols = ['zip', 'loc_key', 'municipality', 'bfs_number', 'canton', 'lat', 'lon', 'language', 'address_share']

# 1. Postcode and locality (largest address share first, then one row per postcode + locality)
plz_loc = df_plz.sort_values('address_share', ascending=False).drop_duplicates(['zip', 'loc_key'])[plz_cols]

# 2. Postcode only (municipality with the largest address share)
plz_only = df_plz.sort_values('address_share', ascending=False).drop_duplicates('zip')[plz_cols].drop(columns='loc_key')

df1 = df.merge(plz_loc, on=['zip', 'loc_key'], how='left', validate='many_to_one')
print('Matched by postcode and locality:', df1['bfs_number'].notna().sum(), 'of', len(df1))

# Fill the rest with the postcode-only match
missing = df1['bfs_number'].isna()
fill = df1.loc[missing, ['zip']].merge(plz_only, on='zip', how='left')
fill.index = df1.index[missing]
for c in ['municipality', 'bfs_number', 'canton', 'lat', 'lon', 'language', 'address_share']:
    df1.loc[missing, c] = fill[c]
print('Matched by postcode only:', fill['bfs_number'].notna().sum())
print('Not matched:', df1['bfs_number'].isna().sum(), '\n')

print(df1.loc[df1['bfs_number'].isna(), ['address', 'zip', 'locality']])

Matched by postcode and locality: 671 of 733
Matched by postcode only: 61
Not matched: 1 

                                   address   zip locality
42  Centralbahnplatz / 10 9/10, 4000 Basel  4000    Basel


### Municipality mergers since 2021 and special postcodes

The postcode directory is from 2026, but the municipality data are from 2021. Some municipalities have merged or changed their number since 2021. For these apartments the **2021 municipality** in which the apartment lies is used:

| Locality | Municipality 2026 (BFS) | Municipality 2021 (BFS) | Reason |
|---|---|---|---|
| Weissbad | Schwende-Rüte (3112) | Schwende (3105) | merger 2022 |
| Marin-Epagnier | Laténa (6513) | La Tène (6461) | merger 2025 |
| Ueken | Herznach-Ueken (4186) | Ueken (4179) | merger 2023 |
| Moutier | Moutier, canton JU (6831) | Moutier, canton BE (700) | change of canton 2026 |
| Neckertal | Neckertal (3396) | Neckertal (3378) | new number |
| Basel, postcode 4000 | – | Basel (2701) | 4000 is a postal-box postcode without addresses |

In [5]:
# New BFS number (2026) -> BFS number of the municipality data (2021)
bfs_2021 = {3112: 3105, 6513: 6461, 4186: 4179, 6831: 700, 3396: 3378}
df1['bfs_number'] = df1['bfs_number'].replace(bfs_2021)

# Postcode 4000 Basel (postal boxes only): use the municipality Basel and the coordinates of postcode 4051
is_4000 = df1['zip'].eq(4000) & df1['bfs_number'].isna()
basel = df_plz.loc[df_plz['zip'].eq(4051)].iloc[0]
df1.loc[is_4000, ['municipality', 'bfs_number', 'canton', 'lat', 'lon', 'language']] = \
    ['Basel', 2701, 'BS', basel['lat'], basel['lon'], 'de']

df1['bfs_number'] = df1['bfs_number'].astype('Int64')
print('Apartments without municipality:', df1['bfs_number'].isna().sum())

Apartments without municipality: 0


## Joining the municipality-level data using .merge()

The key is the BFS number (`bfs_number`), as in Week 9. The merge is `many_to_one`: many apartments, one municipality.

The short column names of the SFSO file are renamed, so that they are easier to understand:

| Name in the file | New name | Meaning |
|---|---|---|
| `bfs_number` | `bfs_number` | official municipality number |
| `bfs_name` | `bfs_name` | official municipality name |
| `pop` | `population` | number of residents |
| `pop_dens` | `population_density` | residents per km² |
| `frg_pct` | `foreigners_pct` | share of foreigners (%) |
| `emp` | `employees` | number of employees (jobs) in the municipality |
| `mean_taxable_income` | `mean_taxable_income` | mean taxable income per taxpayer (CHF) |

In [6]:
df_municip = pd.read_excel('./Data/municipality_data_with_taxable_income.xlsx',
                           sheet_name='data_for_import')

# Rename the short column names
df_municip = df_municip.rename(columns={'pop': 'population',
                                        'pop_dens': 'population_density',
                                        'frg_pct': 'foreigners_pct',
                                        'emp': 'employees'})
print('Municipalities:', df_municip.shape[0])
df_municip.head(3)

Municipalities: 2172


,bfs_number,bfs_name,population,population_density,foreigners_pct,employees,mean_taxable_income
0,1,Aeugst am Albis,1981,250.442478,14.184755,442.0,105013.570634
1,2,Affoltern am Albis,12303,1161.756374,28.700317,6920.0,71059.805603
2,3,Bonstetten,5572,749.932705,16.564968,1014.0,88927.698145


In [7]:
df2 = df1.merge(df_municip[['bfs_number', 'bfs_name', 'population', 'population_density', 'foreigners_pct', 'employees', 'mean_taxable_income']],
                on='bfs_number',
                how='left',
                validate='many_to_one')

print(f'Apartments before merge: {len(df1)}, after merge: {len(df2)}')
print('Apartments without municipality data:', df2['population'].isna().sum())
print('Missing mean taxable income:', df2['mean_taxable_income'].isna().sum())

df2[['address', 'municipality', 'bfs_number', 'bfs_name', 'canton', 'population', 'population_density', 'foreigners_pct', 'mean_taxable_income']].head()

Apartments before merge: 733, after merge: 733
Apartments without municipality data: 0
Missing mean taxable income: 9


,address,municipality,bfs_number,bfs_name,canton,population,population_density,foreigners_pct,mean_taxable_income
0,"Croix Fédérale 44, 2300 La Chaux-de-Fonds",La Chaux-de-Fonds,6421,La Chaux-de-Fonds,NE,37494,672.900215,30.287513,61004.620484
1,"Grüttbachstr. 6, 4542 Luterbach",Luterbach,2527,Luterbach,SO,3488,771.681416,19.323394,61860.606061
2,"Neuhofstr. 20, 5600 Lenzburg",Lenzburg,4201,Lenzburg,AG,10822,956.852343,28.405101,74092.307692
3,"Aathalstr. 19, 8610 Uster",Uster,198,Uster,ZH,35007,1228.746929,23.009684,78620.738763
4,"Rte de Bertigny 2/4, 1700 Fribourg",Fribourg,2196,Fribourg,FR,38197,4116.056034,36.704453,65882.389480


## New variables

- `language_region`: German / French / Italian (from the language of the postcode area; Romansh areas and the bilingual places Biel/Bienne and Domat/Ems, `'multiple'` in swisstopo, are counted as German-speaking, because German is the majority language there)
- `building_age`: 2026 − construction year
- `log_population_density`: logarithm of the population density (the population density is very skewed: from villages to cities)
- `taxable_income_1000`: mean taxable income in 1'000 CHF (easier to read in the regression)

In [8]:
df2['language_region'] = df2['language'].map({'de': 'German', 'rm': 'German', 'multiple': 'German',
                                                'fr': 'French', 'it': 'Italian'})
df2['building_age'] = 2026 - df2['year_built_clean']
df2['log_population_density'] = np.log(df2['population_density'])
df2['taxable_income_1000'] = (df2['mean_taxable_income'] / 1000).round(1)

print(df2['language_region'].value_counts(dropna=False), '\n')
print(df2['canton'].value_counts().head(10))

language_region
German     544
French     150
Italian     39
Name: count, dtype: int64 

canton
BE    132
SG     67
ZH     55
VD     49
AG     45
BL     41
FR     39
TI     39
BS     38
SO     34
Name: count, dtype: int64


## Remove listings that are probably new buildings

These listings say "Neubau", "appartements neufs" or similar (see `05_renovation_details.ipynb`). They do not belong to buildings built before 1990 and are removed.

In [9]:
print('Possible new buildings:')
print(df2.loc[df2['possible_new_build'], ['address', 'year_built_clean', 'renovation_group']], '\n')

df3 = df2.loc[~df2['possible_new_build']].drop(columns=['loc_key', 'address_share', 'language']).reset_index(drop=True)
print('Number of rows after removal:', df3.shape[0])

Possible new buildings:
                                address  year_built_clean   renovation_group
12         Seidenhofstr. 1, 6003 Luzern            1940.0      not renovated
81   J. J. Balmer - Str. 1a, 4053 Basel            1969.0      not renovated
114        Seidenhofstr. 1, 6003 Luzern            1940.0      not renovated
115        Seidenhofstr. 1, 6003 Luzern            1940.0      not renovated
249         Route du Nord 4, 1723 Marly            1988.0  energy renovation
435                     1741 Cottens FR               NaN  energy renovation
440                     1741 Cottens FR               NaN  energy renovation 

Number of rows after removal: 726


### Export data to file

In [10]:
# Check for duplicates
print('Duplicated listing URLs:', df3.duplicated(subset=['listing_url']).sum())

# Count number of rows and columns
print(df3.shape)

# Export apartment data to file
df3.to_csv('./Data/apartments_pre1990_enriched.csv',
           sep=',',
           encoding='utf-8',
           index=False)

Duplicated listing URLs: 0
(726, 50)


### Aggregation: municipality characteristics by renovation group

In [11]:
df3.groupby('renovation_group')[['population_density', 'foreigners_pct', 'mean_taxable_income']].median().round(1)

,population_density,foreigners_pct,mean_taxable_income
renovation_group,,,
energy renovation,907.5,20.3,70203.1
general renovation,901.2,25.8,67561.6
not renovated,995.4,27.9,68263.0


### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [12]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
NT
Windows | 10
Datetime: 2026-11-26 15:45:08
Python Version: 3.11.8
-----------------------------------
